In [2]:
# Graph

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
import random

class AppState(TypedDict):
    input:int
    value:int
    count:int
    response:str

def input_routing_function(state:AppState):
    return state["input"] < 0

def generate_node(state:AppState):
    return {"value" : random.randint(1, 100), "count" : state["count"] + 1}

def routing_function(state:AppState):
    return state["input"] > state["value"]

def terminate_node(state:AppState):
    if "value" not in state.keys():
        return {"response" : f"사용자가 입력한 {state['input']}은(는) 0보다 작습니다."}
    return {"response" : f"{state['count']} 번 반복 실행됐습니다."}

graph = StateGraph(AppState)
graph.add_node("generate", generate_node)
graph.add_node("terminate", terminate_node)

graph.add_conditional_edges(START, input_routing_function, {True:"terminate", False:"generate"})
graph.add_conditional_edges("generate", routing_function, {True:"terminate", False:"generate"})
graph.add_edge("terminate", END)

app = graph.compile()

In [ ]:
# invoke
app.invoke({"input":10, "count":0})

In [ ]:
# ainvoke
result = await app.ainvoke({"input":10, "count":0})
print(result)

In [ ]:
# batch
works = [
    {"input":10, "count":0},
    {"input":20, "count":0},
    {"input":30, "count":0}
    ]
app.batch(works)

In [ ]:
# stream : 기본
for result in app.stream({"input":10, "count":0}):
    print(result)

In [ ]:
# stream : values
mode = "values"
for result in app.stream({"input":10, "count":0}, stream_mode=mode):
    print(result)

In [ ]:
# stream : updates
mode = "updates"
for result in app.stream({"input":10, "count":0}, stream_mode=mode):
    print(result)

In [ ]:
# stream : debug
mode = "debug"
for result in app.stream({"input":10, "count":0}, stream_mode=mode):
    print(result)

In [4]:
# stream : messages
mode = "messages"
for chunk in app.stream({"input":10, "count":0}, stream_mode=mode):
    message_chunk, metadata = chunk
    print(message_chunk.conetent)

In [ ]:
# astream : a
mode = "updates"
async for result in app.astream({"input":10, "count":0}, stream_mode=mode):
    print(result)